In [ ]:
from pymilvus import AnnSearchRequest, Function, FunctionType, MilvusClient,RRFRanker

import provider

embedding_config = provider.config["qwen-embedding"]
embedding_client = embedding_config.client

Initializing OpenAI clients for all providers...


In [3]:
db = MilvusClient("./milvus_demo.db")
collection_name = "my_collection"   
db.load_collection(collection_name=collection_name)
print(db.get_collection_stats(collection_name=collection_name))

{'row_count': 8}


In [4]:
queries=["星云AI有哪些模型","星云AI模型费用"]
embedding_result = embedding_client.embeddings.create(  # type: ignore
        model=embedding_config.model,
        input=queries,
    ).data
queries_embeddings = [item.embedding for item in embedding_result]
print([len(item) for item in queries_embeddings])

[1024, 1024]


In [5]:
res = db.search(  collection_name=collection_name,
    filter="category == '技术'",
     anns_field= "text_dense",
    data=queries_embeddings,
    limit=4,
    output_fields=["text"])
for i, hits in enumerate(res):
    print(f"TopK results for query {i} {queries[i]}:")
    for hit in hits:
        print(hit)

TopK results for query 0 星云AI有哪些模型:
{'id': 5, 'distance': 0.2832794189453125, 'entity': {'id': 5, 'text': '该片段出自《星云AI平台API调用规范与限流策略 v4.2》第3节“模型推理接口”，列出了端点 `/v4/inference` 当前支持的三个具体模型名称。\n支持模型列表： nebula-chat-pro, nebula-code-v2, nebula-vision-lite。'}}
{'id': 6, 'distance': 0.4435539245605469, 'entity': {'id': 6, 'text': '该片段出自《星云AI平台API调用规范与限流策略 v4.2》文档的第3节“模型推理接口”，具体位于对 `/v4/inference` 端点支持的模型列表说明中。其核心内容是澄清 `nebula-vision-lite` 模型的访问权限限制，明确指出该模型仅对企业租户开放，并消除了关于错误响应的歧义：标准租户若尝试调用此受限模型，系统将返回 HTTP 403 禁止访问错误，而非限流策略中常见的 HTTP 429 错误。\n注意： nebula-vision-lite 仅对企业租户开放。标准租户调用该模型将返回HTTP 403而非429。'}}
{'id': 1, 'distance': 0.4465900659561157, 'entity': {'id': 1, 'text': '这段内容出自《星云AI平台API调用规范与限流策略 v4.2》文档的第1节“概述”，主要阐述了该文档旨在定义平台所有对外接口的调用标准，并明确指出自2026年6月1日起，旧的v3.0鉴权方式已被废弃，所有请求强制要求使用HMAC-SHA256签名进行身份验证。\n本文档定义了星云AI平台所有对外接口的调用标准。自2026年6月1日起，旧版v3.0鉴权方式已彻底废弃，所有请求必须使用HMAC-SHA256签名。'}}
{'id': 8, 'distance': 0.45391976833343506, 'entity': {'id': 8, 'text': '该片段出自《星云AI平台API调用规范与v4.2》文档的第4节“计费说明”，旨在阐述模型推理接口的具体

I0707 11:02:08.297970 9759399 chttp2_transport.cc:1376] ipv4:127.0.0.1:55416: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E0707 11:02:08.297990 9759399 chttp2_transport.cc:1408] ipv4:127.0.0.1:55416: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


In [6]:
res = db.search(
    collection_name='my_collection', 
    data=queries,
    anns_field='text_sparse',
    output_fields=['text'], 
    limit=4,
)

for i, hits in enumerate(res):
    print(f"TopK results for query {i} {queries[i]}:")
    for hit in hits:
        print(hit)

Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/w5/ytwlrs850bjgxr__l0gglxwr0000gn/T/jieba.cache
Loading model cost 0.386 seconds.
Prefix dict has been built successfully.


TopK results for query 0 星云AI有哪些模型:
{'id': 6, 'distance': -1.429341197013855, 'entity': {'id': 6, 'text': '该片段出自《星云AI平台API调用规范与限流策略 v4.2》文档的第3节“模型推理接口”，具体位于对 `/v4/inference` 端点支持的模型列表说明中。其核心内容是澄清 `nebula-vision-lite` 模型的访问权限限制，明确指出该模型仅对企业租户开放，并消除了关于错误响应的歧义：标准租户若尝试调用此受限模型，系统将返回 HTTP 403 禁止访问错误，而非限流策略中常见的 HTTP 429 错误。\n注意： nebula-vision-lite 仅对企业租户开放。标准租户调用该模型将返回HTTP 403而非429。'}}
{'id': 5, 'distance': -1.4275909662246704, 'entity': {'id': 5, 'text': '该片段出自《星云AI平台API调用规范与限流策略 v4.2》第3节“模型推理接口”，列出了端点 `/v4/inference` 当前支持的三个具体模型名称。\n支持模型列表： nebula-chat-pro, nebula-code-v2, nebula-vision-lite。'}}
{'id': 7, 'distance': -0.8351560235023499, 'entity': {'id': 7, 'text': '该片段出自《星云AI平台API调用规范与v4.2》文档第3节“模型推理接口 /v4/inference”，主要说明了该接口请求超时的配置规则，指出默认超时时间为30秒，最大允许设置为120秒，且任何超过此阈值的请求将被网关直接拦截而不会进入后端推理队列。\n超时设置： 默认30s，最大可配置120s。超过120s的请求会被网关直接拒绝，不会进入推理队列。'}}
{'id': 8, 'distance': -0.7139049172401428, 'entity': {'id': 8, 'text': '该片段出自《星云AI平台API调用规范与v4.2》文档的第4节“计费说明”，旨在阐述模型推理接口的具体收费标准及免责情形。其中，“推理费用”特指针对第3节

In [7]:

search_param_1 = {
    "data": queries_embeddings,
    "anns_field": "text_dense",
    "param": {"nprobe": 3},
    "limit": 4
}

search_param_2 = {
    "data": queries,
    "anns_field": "text_sparse",
    "param": {},
    "limit": 4
}

res = db.hybrid_search(
    collection_name=collection_name,
    filter="category == '技术'",
    reqs=[AnnSearchRequest(**search_param_1), AnnSearchRequest(**search_param_2)],
    ranker=RRFRanker(),
    limit=8,
    output_fields=["text"]
)

for i, hits in enumerate(res):
    print(f"TopK results for query {i} {queries[i]}:")
    for hit in hits:
        print(hit)

TopK results for query 0 星云AI有哪些模型:
{'id': 5, 'distance': 0.032522473484277725, 'entity': {'id': 5, 'text': '该片段出自《星云AI平台API调用规范与限流策略 v4.2》第3节“模型推理接口”，列出了端点 `/v4/inference` 当前支持的三个具体模型名称。\n支持模型列表： nebula-chat-pro, nebula-code-v2, nebula-vision-lite。'}}
{'id': 6, 'distance': 0.032522473484277725, 'entity': {'id': 6, 'text': '该片段出自《星云AI平台API调用规范与限流策略 v4.2》文档的第3节“模型推理接口”，具体位于对 `/v4/inference` 端点支持的模型列表说明中。其核心内容是澄清 `nebula-vision-lite` 模型的访问权限限制，明确指出该模型仅对企业租户开放，并消除了关于错误响应的歧义：标准租户若尝试调用此受限模型，系统将返回 HTTP 403 禁止访问错误，而非限流策略中常见的 HTTP 429 错误。\n注意： nebula-vision-lite 仅对企业租户开放。标准租户调用该模型将返回HTTP 403而非429。'}}
{'id': 8, 'distance': 0.03125, 'entity': {'id': 8, 'text': '该片段出自《星云AI平台API调用规范与v4.2》文档的第4节“计费说明”，旨在阐述模型推理接口的具体收费标准及免责情形。其中，“推理费用”特指针对第3节定义的`/v4/inference`接口所产生的成本，明确了输入与输出的单价差异，并规定缓存命中及参数校验失败（HTTP 400）的情况不纳入计费范围。\n推理费用按Token计费，输入¥0.02/千Token，输出¥0.06/千Token。缓存命中（Cache Hit）的Token不计费。若请求因参数校验失败（HTTP 400）被拒绝，不产生任何费用。'}}
{'id': 1, 'distance': 0.01587301678955555, 'entity': {'id': 1, 'text': '这段内容出自《星云

I0707 11:06:36.852159 9759393 chttp2_transport.cc:1376] ipv4:127.0.0.1:55416: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E0707 11:06:36.852197 9759393 chttp2_transport.cc:1408] ipv4:127.0.0.1:55416: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


In [8]:
import dashscope
from http import HTTPStatus

dashscope.base_http_api_url = "https://dashscope.aliyuncs.com/api/v1"


def text_rerank(query:str, chunks:list[str])-> list[int]:
    resp = dashscope.TextReRank.call(
        model="qwen3-rerank",
        query=query,
        documents=chunks,
        top_n=10,
        instruct="Given a web search query, retrieve relevant passages that answer the query."
    )
    if resp.status_code == HTTPStatus.OK:
        print(resp)
        return [i['index'] for i in resp.output.results]
    else:
        print(resp)
        return []

In [9]:
result = []
for i,hits in enumerate(res):
    chunks=[hits[i]['entity']['text'] for i in range(len(hits))]
    result_idx=text_rerank(queries[i], chunks)
    result.append([hits[i]['entity'] for i in result_idx])
for i, hits in enumerate(result):
    print(f"TopK results for query {i} {queries[i]}:")
    for hit in hits:
        print(hit)

{"status_code": 200, "request_id": "54135689-fdc6-9153-92a9-e97ba2163b5c", "code": "", "message": "", "output": {"results": [{"index": 0, "relevance_score": 0.9440524937841727, "document": null}, {"index": 1, "relevance_score": 0.8083460240276599, "document": null}, {"index": 4, "relevance_score": 0.5841706746488604, "document": null}, {"index": 2, "relevance_score": 0.43760163318273465, "document": null}, {"index": 3, "relevance_score": 0.42526920257701006, "document": null}]}, "usage": {"total_tokens": 757}}
{"status_code": 200, "request_id": "b734f3bd-b536-9265-98bc-0149e35f80ef", "code": "", "message": "", "output": {"results": [{"index": 0, "relevance_score": 0.7834657693234912, "document": null}, {"index": 1, "relevance_score": 0.5781629036527827, "document": null}, {"index": 5, "relevance_score": 0.47713584981183715, "document": null}, {"index": 2, "relevance_score": 0.4707717492718393, "document": null}, {"index": 3, "relevance_score": 0.4632338113433238, "document": null}, {"i